# Ensemble wordwise contextuel (Transformer-Day) - Execution pas a pas

Ce notebook reprend `runoff/mains.py` en version pas a pas.
Il accepte uniquement des modeles entraines via le notebook precedent (Transformer + DayInputLayer).

Modifie les chemins dans **Etape 0 - Configuration** avant execution.



## Etape 0 - Configuration (chemins, modele, evaluation)


In [1]:
# Imports (meme ordre general que le script)
import os
import math
import json
from datetime import datetime
from typing import List, Dict, Tuple, Optional

import numpy as np
import h5py
import torch
import pandas as pd

import sys
from pathlib import Path

# Racine du projet = dossier contenant ce notebook
RUNOFF_DIR = Path.cwd()


from transformersday import predict_ids_from_tensor as tf_day_predict_ids


In [2]:
# --- Chemins principaux ---
DATA_ROOT = str(RUNOFF_DIR / "data")
SPLIT_CSV = str(RUNOFF_DIR / "data" / "t15_sessions_random_split.csv")
IN_CSV = str(RUNOFF_DIR / "data" / "phrases_phonemes_merged_h5_cmu.csv")

# --- Cles HDF5 attendues ---
NEURAL_KEY_PRIMARY = "input_features"
NEURAL_KEY_FALLBACK = "neural_features"
LABELS_KEY = "seq_class_ids"
TEXT_KEYS = ("sentence_label", "transcriptions", "phrase_label")

# --- Hyperparams generaux ---
N_FEATURES = 512
BLANK_ID = 0
BOUNDARY_ID = 40
SEED = 1337
MAX_PHONEMES_FOR_WER = 30

# Optionnel: cache lexique + LM (mettre None pour desactiver)
LEXICON_CACHE = None


# --- Modeles ---
PRETRAINED_ROOT = RUNOFF_DIR / "pretrained models"


MODEL_SPECS = [
    {
        "name": "pretrained1",
        "config_json": PRETRAINED_ROOT/r"20251130-044021/config.json",
        "checkpoint": PRETRAINED_ROOT/r"20251130-044021/model_bestPER.pt",
    },
    {
        "name": "pretrained2",
        "config_json": PRETRAINED_ROOT/r"20251130-044246/config.json",
        "checkpoint": PRETRAINED_ROOT/r"20251130-044246/model_bestPER.pt",
    },
    {
        "name": "trained3",
        "config_json": r"C:\code\brain2text_bundle\brain2text_project\outputs\runs\20260207-031115\config.json",
        "checkpoint": r"C:\code\brain2text_bundle\brain2text_project\outputs\runs\20260207-031115\model_bestPER.pt",
    },
    #add othe models by adding the path of a config and model in .pt with the same format
]


## Etape 1 - Outils generaux (seed, texte, sequences)


In [3]:
def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def normalize_text(text: str) -> str:
    text = str(text).lower()
    # Remplace les caracteres non-alphanumeriques par des espaces
    import re
    text = re.sub(r"[^a-z0-9\s']+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def parse_id_sequence(s: str, blank_id: int = 0) -> List[int]:
    ids: List[int] = []
    if s is None:
        return ids
    text = str(s).strip()
    if not text:
        return ids
    text = text.replace(",", " ")
    for tok in text.split():
        try:
            v = int(tok)
        except Exception:
            continue
        if v == blank_id:
            continue
        ids.append(v)
    return ids


def split_on_boundary(seq: List[int], boundary_id: int = BOUNDARY_ID, blank_id: int = BLANK_ID) -> List[List[int]]:
    # Decoupe une sequence d'IDs phonemes en liste de mots (liste de sequences), en utilisant boundary_id.
    words: List[List[int]] = []
    current_word: List[int] = []
    for x in seq:
        xi = int(x)
        if xi == blank_id:
            continue
        if xi == boundary_id:
            if current_word:
                words.append(current_word)
                current_word = []
        else:
            current_word.append(xi)
    if current_word:
        words.append(current_word)
    return words


## Etape 2 - Lexique + LM (n-gram)


In [4]:
class LexiconTrieNode:  # Noeud d'un trie (prefix tree) pour phonemes
    def __init__(self):  # Initialise le noeud
        self.children: Dict[int, "LexiconTrieNode"] = {}  # phoneme_id -> sous-noeud
        self.word_ends: List[str] = []  # mots qui se terminent ici


class LexiconTrie:  # Trie pour associer sequence de phonemes -> mots
    def __init__(self):  # Initialise le trie
        self.root = LexiconTrieNode()  # racine vide du trie

    def insert(self, word: str, phoneme_seq: List[int]):  # Ajoute un mot et sa sequence
        node = self.root  # depart a la racine
        for ph in phoneme_seq:  # avance phoneme par phoneme
            if ph not in node.children:  # cree le noeud si absent
                node.children[ph] = LexiconTrieNode()  # nouveau noeud enfant
            node = node.children[ph]  # descend d'un niveau
        node.word_ends.append(word)  # marque la fin du mot


def build_lexicon_and_lm(
    csv_path: str,
    boundary_id: int = BOUNDARY_ID,
    blank_id: int = BLANK_ID,
    ngram_order: int = 3,
) -> Tuple[LexiconTrie, Dict]:
    # Construit le lexique (trie) et les stats n-gram a partir d'un CSV
    df = pd.read_csv(csv_path)

    phon_col = "phonemes"
    text_col = "phrase"


    # Initialise le trie qui mappe sequence de phonemes -> mot
    lexicon = LexiconTrie()
    # Stocke toutes les phrases tokenisees (liste de mots) pour calculer les n-grams
    tokenized_sentences: List[List[str]] = []
    # Ensemble des mots uniques observes dans le corpus
    vocab = set()
    # Compteur du nombre total de mots (occurrences, pas uniques)
    total_words = 0

    # Parcourt chaque ligne du CSV pour construire lexique + stats de corpus
    for _, row in df.iterrows():
        # Convertit la colonne phonemes en liste d'IDs (en retirant les blanks)
        phon_seq = parse_id_sequence(row[phon_col], blank_id=blank_id)
        # Normalise le texte (minuscule + nettoyage)
        text = normalize_text(row[text_col])
        # Ignore les lignes incompletes
        if not phon_seq or not text:
            continue
        # Decoupe la sequence phonemes en "mots phonemiques" via le token boundary
        word_phonemes = split_on_boundary(phon_seq, boundary_id=boundary_id, blank_id=blank_id)
        # Decoupe la phrase texte en mots
        words = text.split()
        # Garde la phrase tokenisee pour le calcul des n-grams
        if words:
            tokenized_sentences.append(words)
        for w, ph_seq in zip(words, word_phonemes):        # Aligne chaque mot texte avec sa sequence phonemique et alimente le lexique

            if w and ph_seq:
                lexicon.insert(w, ph_seq)                # Ajoute la correspondance phonemes -> mot dans le trie

                vocab.add(w)                # Ajoute le mot au vocabulaire unique

                total_words += 1

    # Initialise le conteneur principal des stats n-gram
    # Exemple: ngram_stats[1] pour les unigrams, ngram_stats[2] pour les bigrams, etc.
    ngram_stats = {}
    for n in range(1, ngram_order + 1):
        ngram_stats[n] = {}

    for words in tokenized_sentences:    # Parcourt chaque phrase tokenisee pour compter les occurrences
        L = len(words)
        for i in range(L):        # Parcourt chaque position de mot dans la phrase
            w = words[i]          # Mot courant a la position i
            ngram_stats[1][w] = ngram_stats[1].get(w, 0) + 1     # Incremente le compteur unigram du mot courant

            # Construit aussi les n-grams de taille 2..ngram_order
            for n in range(2, ngram_order + 1):
                if i - (n - 1) < 0: # Pas assez de contexte a gauche pour former ce n-gram
                    break
                # Extrait le n-gram (fenetre de mots) qui se termine a i
                gram = tuple(words[i - (n - 1): i + 1])
                # Incremente la frequence de ce n-gram
                ngram_stats[n][gram] = ngram_stats[n].get(gram, 0) + 1

    # Metadonnees utiles pour le scoring (evite division par zero avec max(1, ...))
    ngram_stats["vocab"] = max(1, len(vocab))
    # Nombre total de mots observes dans le corpus
    ngram_stats["total_words"] = max(1, total_words)
    # Ordre maximal des n-grams construits
    ngram_stats["max_n"] = ngram_order
    return lexicon, ngram_stats


def lexicon_lookup(phoneme_seq: List[int], lexicon: LexiconTrie) -> List[str]:
    # Retourne les mots candidats associes a une sequence de phonemes.
    node = lexicon.root
    for ph in phoneme_seq:
        if ph not in node.children:
            return []
        node = node.children[ph]
    return list(node.word_ends)


def score_unigram(word: str, stats: Dict, alpha: float = 0.35) -> float:
    if word is None:
        count = 0
    else:
        count = stats.get(1, {}).get(word, 0)
    V = stats.get("vocab", 1)
    total = stats.get("total_words", 1)
    prob = (count + alpha) / (total + alpha * V)
    return math.log(prob)


def score_conditional(word: str, history: List[str], stats: Dict, alpha: float = 0.35) -> float:
    # Score log-probabiliste de `word` conditionne par le contexte precedent.
    if not history:
        return 0.0  # Aucun contexte: pas de contrainte conditionnelle.
    h = history[-2:]  # On garde au plus 2 mots d'historique (mode trigram).
    h_len = len(h)
    V = stats.get("vocab", 1)  # Taille du vocabulaire pour le lissage add-alpha.
    if h_len == 1:
        # Cas bigram: P(word | prev) = count(prev, word) / count(prev).
        prev = h[0]
        denom = stats.get(1, {}).get(prev, 0)
        numer = stats.get(2, {}).get((prev, word), 0)
    else:
        # Cas trigram: P(word | w_{t-2}, w_{t-1}) = count(w_{t-2}, w_{t-1}, word) / count(w_{t-2}, w_{t-1}).
        prev_bigram = (h[0], h[1])
        denom = stats.get(2, {}).get(prev_bigram, 0)
        numer = stats.get(3, {}).get((h[0], h[1], word), 0)

    if denom > 0:
        # Lissage add-alpha pour eviter les probabilites nulles sur des n-grams rares.
        prob = (numer + alpha) / (denom + alpha * V)
    else:
        prob = 1.0 / V  # Contexte jamais vu: fallback uniforme.
    return math.log(prob)  # Log pour additionner les scores de facon numeriquement stable.


def load_or_build_lexicon(
    csv_path: str,
    cache_path: Optional[str],
    boundary_id: int,
    blank_id: int,
    ngram_order: int = 3,
) -> Tuple[LexiconTrie, Dict]:
    if cache_path and os.path.isfile(cache_path):
        import pickle
        with open(cache_path, "rb") as f:
            cached = pickle.load(f)
        # Supporte anciens formats de cache.
        if isinstance(cached, tuple):
            if len(cached) >= 2:
                lexicon, ngram_stats = cached[0], cached[1]
            else:
                raise ValueError("Cache lexicon invalide: tuple trop court")
        elif isinstance(cached, dict):
            if "lexicon" in cached and "ngram_stats" in cached:
                lexicon = cached["lexicon"]
                ngram_stats = cached["ngram_stats"]
            else:
                raise ValueError("Cache lexicon invalide: cles manquantes")
        else:
            raise ValueError("Cache lexicon invalide: format inconnu")
        # Retourne le lexique (trie) et les stats n-gram pretes pour le vote contextuel
        return lexicon, ngram_stats

    lexicon, ngram_stats = build_lexicon_and_lm(
        csv_path,
        boundary_id=boundary_id,
        blank_id=blank_id,
        ngram_order=ngram_order,
    )

    if cache_path:
        import pickle
        with open(cache_path, "wb") as f:
            pickle.dump((lexicon, ngram_stats), f)

    # Retourne le lexique (trie) et les stats n-gram pretes pour le vote contextuel
    return lexicon, ngram_stats


## Etape 3 - Lecture HDF5 (trials)


In [5]:
def load_train_eval_dates_from_csv(split_csv: str) -> List[str]:
    import csv
    train_dates = []
    with open(split_csv, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            date = row.get("date", "").strip()
            typ = row.get("type", "").strip().lower()
            if typ != "train":
                continue
            if not date:
                continue
            folder_name = "t15." + date.replace("-", ".")
            train_dates.append(folder_name)
    return train_dates


def build_h5_paths_from_dates(root: str, date_folders: List[str], filename: str = "data_val.hdf5") -> List[str]:
    paths = []
    for folder in date_folders:
        full_path = os.path.join(root, folder, filename)
        if not os.path.isfile(full_path):
            print(f"[WARN] Fichier absent : {full_path}")
            continue
        paths.append(full_path)
    return paths


def build_eval_index_from_hdf5(h5_paths: List[str]) -> List[Dict]:
    index = []
    gid = 0
    for file_path in h5_paths:
        with h5py.File(file_path, "r") as f:
            trial_keys = [k for k in f.keys() if k.startswith("trial_") and k[6:].isdigit()]
        print(f"[EVAL] {os.path.basename(file_path)} -> {len(trial_keys)} trials")
        for k in trial_keys:
            index.append({
                "global_id": gid,
                "file_path": file_path,
                "trial_key": k,
                "trial_num": int(k[6:])
            })
            gid += 1
    print(f"[EVAL] Total trials : {len(index)}")
    return index


def _decode_text_like(obj) -> str:
    if obj is None:
        return ""
    try:
        import numpy as _np
        if isinstance(obj, _np.ndarray):
            if obj.dtype.kind in ("S", "U"):
                if obj.size == 1:
                    return str(obj.reshape(-1)[0]).strip()
                return " ".join(str(x) for x in obj.reshape(-1).tolist()).strip()
            if obj.dtype.kind in ("i", "u"):
                try:
                    chars = [chr(int(x)) for x in obj.reshape(-1).tolist() if int(x) != 0]
                    return "".join(chars).strip()
                except Exception:
                    pass
            return str(obj).strip()
    except Exception:
        pass
    if isinstance(obj, (bytes, bytearray)):
        try:
            return obj.decode("utf-8", errors="ignore").strip()
        except Exception:
            return str(obj).strip()
    if isinstance(obj, (list, tuple)):
        try:
            if obj and all(isinstance(x, (int, np.integer)) for x in obj):
                chars = [chr(int(x)) for x in obj if int(x) != 0]
                return "".join(chars).strip()
        except Exception:
            pass
        return " ".join(str(x) for x in obj).strip()
    return str(obj).strip()


def read_text_from_group(g: h5py.Group) -> str:
    for key in TEXT_KEYS:
        if key in g:
            try:
                txt = _decode_text_like(g[key][:])
                if txt:
                    return txt
            except Exception:
                pass
    for key in TEXT_KEYS:
        if key in g.attrs:
            try:
                txt = _decode_text_like(g.attrs.get(key))
                if txt:
                    return txt
            except Exception:
                pass
    return ""


def read_neural_from_group(g: h5py.Group) -> np.ndarray:
    if NEURAL_KEY_PRIMARY in g:
        x = g[NEURAL_KEY_PRIMARY][:]
    elif NEURAL_KEY_FALLBACK in g:
        x = g[NEURAL_KEY_FALLBACK][:]
    else:
        raise KeyError(
            f"Aucune cle neurale trouvee dans le trial (attendu '{NEURAL_KEY_PRIMARY}' ou '{NEURAL_KEY_FALLBACK}')."
        )
    if x.ndim != 2:
        raise ValueError("Les features neuronales doivent etre 2D.")
    if x.shape[0] == N_FEATURES and x.shape[1] != N_FEATURES:
        x = x.T
    return x


def load_trial(item: Dict) -> Dict:
    file_path = item["file_path"]
    trial_key = item["trial_key"]
    with h5py.File(file_path, "r") as f:
        g = f[trial_key]
        x_np = read_neural_from_group(g)
        y_np = g[LABELS_KEY][:] if LABELS_KEY in g else np.zeros((0,), dtype=np.int64)
        session = _decode_text_like(g.attrs.get("session", ""))
        phrase = read_text_from_group(g)
        block_num = g.attrs.get("block_num", None)
        trial_num_attr = g.attrs.get("trial_num", None)
    x = torch.from_numpy(x_np).float()
    y = torch.from_numpy(y_np).long()
    trial_num = item.get("trial_num", -1)
    if trial_num_attr is not None:
        try:
            trial_num = int(trial_num_attr)
        except Exception:
            pass
    return {
        "trial_num": trial_num,
        "block_num": int(block_num) if block_num is not None else None,
        "x": x,
        "y": y,
        "session": session,
        "phrase": _decode_text_like(phrase),
    }


## Etape 4 - Metriques (PER / WER)


In [6]:
def levenshtein_distance(a: List[int], b: List[int]) -> int:
    dp = list(range(len(b) + 1))
    for i, x in enumerate(a, start=1):
        prev = dp[0]
        dp[0] = i
        for j, y in enumerate(b, start=1):
            cur = dp[j]
            dp[j] = prev if x == y else 1 + min(prev, dp[j], dp[j - 1])
            prev = cur
    return dp[-1]


def phoneme_error_rate(pred: List[int], ref: List[int], blank_id: int = BLANK_ID) -> float:
    ref_filtered = [int(i) for i in ref if int(i) != blank_id]
    if not ref_filtered:
        return 0.0 if not pred else float("nan")
    dist = levenshtein_distance(pred, ref_filtered)
    return dist / len(ref_filtered)


def word_levenshtein(a: List[str], b: List[str]) -> int:
    dp = list(range(len(b) + 1))
    for i, x in enumerate(a, start=1):
        prev = dp[0]
        dp[0] = i
        for j, y in enumerate(b, start=1):
            cur = dp[j]
            dp[j] = prev if x == y else 1 + min(prev, dp[j], dp[j - 1])
            prev = cur
    return dp[-1]


def compute_wer(hyp_text: str, ref_text: str) -> float:
    hyp_words = normalize_text(hyp_text).split()
    ref_words = normalize_text(ref_text).split()
    if not ref_words:
        return 0.0 if not hyp_words else float("nan")
    dist = word_levenshtein(hyp_words, ref_words)
    return dist / len(ref_words)


## Etape 5 - Ensemble par vote contextuel


In [7]:
def ensemble_word_context_vote(
    pred_word_lists: List[List[List[int]]],
    lexicon: LexiconTrie,
    ngram_stats: Dict,
) -> Tuple[List[int], List[str]]:
    # Fusionne plusieurs sorties modeles en choisissant le meilleur mot a chaque position.
    M = len(pred_word_lists)
    if M == 0:
        return [], []  # Aucun modele a combiner.

    # Estime la longueur de phrase cible (en nombre de mots).
    n_words_list = [len(words) for words in pred_word_lists]
    avg_words = sum(n_words_list) / float(len(n_words_list)) if n_words_list else 0.0
    max_n = max(n_words_list) if n_words_list else 0

    K = int(round(avg_words))
    if max_n > 0:
        K = max(1, min(K, max_n))  # Borne K entre 1 et la plus longue prediction.
    else:
        K = 0

    chosen_words: List[str] = []
    chosen_phonemes: List[List[int]] = []

    # Selection gloutonne mot par mot avec un score de modele de langue.
    for i in range(K):
        candidates: List[Tuple[List[int], str, float]] = []
        for seq_idx, word_list in enumerate(pred_word_lists):
            if i < len(word_list):
                phon_seq = word_list[i]
                word_texts = lexicon_lookup(phon_seq, lexicon)  # Phonemes -> un ou plusieurs mots.
                if not word_texts:
                    word_texts = [None]  # Sequence inconnue du lexique.
                for w_text in word_texts:
                    # Score total = prior unigram + score conditionnel selon l'historique deja choisi.
                    s1 = score_unigram(w_text, ngram_stats, alpha=0.35)
                    s2 = score_conditional(
                        w_text if w_text is not None else "<UNK>",
                        chosen_words,
                        ngram_stats,
                        alpha=0.35,
                    )
                    total_score = s1 + s2
                    candidates.append((phon_seq, w_text if w_text is not None else "<UNK>", total_score))

        if not candidates:
            break  # Plus aucun mot candidat a cette position.

        candidates.sort(key=lambda x: x[2], reverse=True)
        best_phon_seq, best_word, _ = candidates[0]  # On garde le candidat le mieux score.
        chosen_phonemes.append(best_phon_seq)
        chosen_words.append(best_word)

    # Recompose la sequence phonemique finale (avec separateur de mots).
    final_phoneme_seq: List[int] = []
    for ph_seq in chosen_phonemes:
        final_phoneme_seq.extend(ph_seq)
        final_phoneme_seq.append(BOUNDARY_ID)

    return final_phoneme_seq, chosen_words


## Etape 6 - Prediction (Transformer-Day uniquement)


In [8]:
def transformers_day_prediction(x: torch.Tensor, pathmodel: str, pathjson: str, session: str) -> List[int]:
    return tf_day_predict_ids(x, config_json_path=pathjson, checkpoint_path=pathmodel, session=session)


def validate_model_specs(specs: List[Dict]):
    for spec in specs:
        if not os.path.isfile(spec.get("config_json", "")):
            raise FileNotFoundError(f"config_json introuvable: {spec.get('config_json')}")
        if not os.path.isfile(spec.get("checkpoint", "")):
            raise FileNotFoundError(f"checkpoint introuvable: {spec.get('checkpoint')}")


## Etape 7 - Creation / chargement du lexique


In [9]:
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilise :", device)

assert os.path.exists(DATA_ROOT), f"Chemin DATA_ROOT introuvable: {DATA_ROOT}"
assert os.path.exists(SPLIT_CSV), f"Fichier CSV split introuvable: {SPLIT_CSV}"
assert os.path.exists(IN_CSV), f"CSV lexique introuvable: {IN_CSV}"

validate_model_specs(MODEL_SPECS)

print("Chargement du lexique et des statistiques n-gram...")
lexicon, ngram_stats = load_or_build_lexicon(
    IN_CSV,
    cache_path=LEXICON_CACHE,
    boundary_id=BOUNDARY_ID,
    blank_id=BLANK_ID,
    ngram_order=3,
)
print("Lexique et LM charges.")


# A ce stade, lexicon et ngram_stats sont pret pour le vote contextuel.


Device utilise : cuda
Chargement du lexique et des statistiques n-gram...
Lexique et LM charges.


## Etape 8 - Lecture du lexique et vote contextuel


In [10]:
# Charge les dates d'evaluation depuis le CSV de split.
train_dates = load_train_eval_dates_from_csv(SPLIT_CSV)
# Construit les chemins vers les HDF5 d'evaluation.
h5_paths = build_h5_paths_from_dates(DATA_ROOT, train_dates, filename="data_val.hdf5")
# Construit l'index des trials a evaluer.
eval_index = build_eval_index_from_hdf5(h5_paths)


# Accumulateurs de metriques et dump detaille.
per_list: List[float] = []
wer_list: List[float] = []
results_dump: List[Dict] = []

print("[ENSEMBLE MODE] wordwise contextuel (vote par mot avec lexique + contexte)")
# Boucle principale sur les trials d'evaluation.
for item in eval_index:
    # Charge un trial et prepare l'entree modele.
    batch = load_trial(item)
    x = batch["x"].unsqueeze(0).to(device)
    y = batch["y"]
    # Metadonnees pour affichage et suivi.
    session = batch.get("session", "")
    phrase = batch.get("phrase", "")
    block_num = batch.get("block_num", None)
    trial_num = batch.get("trial_num", item.get("trial_num", -1))

    # Retire les blanks et saute si sequence cible vide.
    targets = y[y != BLANK_ID].tolist()
    if not targets:
        continue

    # Resume du trial.
    print("" + "=" * 80)
    print(
        f"TRIAL global_id={item['global_id']} | file={os.path.basename(item['file_path'])} "
        f"| key={item['trial_key']} | session={session}"
        + (f" | block_num={block_num}" if block_num is not None else "")
        + (f" | trial_num={trial_num}" if trial_num is not None else "")
    )
    if phrase:
        print(f"PHRASE LABEL: {phrase}")
    else:
        print("PHRASE LABEL: [indisponible]")
    print(f"Targets (len={len(targets)}): {targets}")

    # Accumulateurs par modele.
    pred_sequences: List[List[int]] = []
    model_names: List[str] = []
    model_word_lists: List[List[List[int]]] = []

    # Inference de chaque modele de l'ensemble.
    for spec in MODEL_SPECS:
        name = spec.get("name", "model-")
        ckpt_path = spec["checkpoint"]
        cfg_path = spec["config_json"]

        pred_ids = transformers_day_prediction(x, ckpt_path, cfg_path, session=session)
        # Normalise la sortie en liste Python d'entiers.
        if torch.is_tensor(pred_ids):
            pred_ids = pred_ids.tolist()  # Tensor -> list
        else:
            pred_ids = [int(x) for x in pred_ids]  # Iterable -> int
        Lm = len(pred_ids)

        # Sauvegarde la sequence brute et le nom du modele.
        pred_sequences.append(pred_ids)
        model_names.append(name)

        # Decoupe en mots via le separateur de boundaries.
        word_list = split_on_boundary(pred_ids, boundary_id=BOUNDARY_ID, blank_id=BLANK_ID)
        model_word_lists.append(word_list)
        print(f"[MODEL {name}] len={Lm} -> preds={pred_ids}")

    # Stats du nombre de mots predits par modele.
    word_counts = [len(wlist) for wlist in model_word_lists]
    avg_words = sum(word_counts) / float(len(word_counts)) if word_counts else 0.0
    max_words = max(word_counts) if word_counts else 0
    target_words = len(phrase.split()) if phrase else None
    print(
        f"[WORD COUNT] models_pred={word_counts} -> avg~{avg_words:.1f}, max={max_words}"
        + (f" | target_words={target_words}" if target_words is not None else "")
    )

    # Vote word-wise avec lexique + contexte n-gram.
    ensemble_phonemes, ensemble_words = ensemble_word_context_vote(
        model_word_lists, lexicon, ngram_stats
    )
    print(f"[ENSEMBLE WORD CTX] len={len(ensemble_phonemes)} preds={ensemble_phonemes}")

    # Calcule le PER du trial.
    per = phoneme_error_rate(ensemble_phonemes, targets, blank_id=BLANK_ID)
    print(f"[TRIAL PER] {per:.4f}")
    per_list.append(per)

    # Calcule le WER si la phrase est dispo.
    wer = None
    if phrase:
        hyp_text = " ".join(ensemble_words)
        wer = compute_wer(hyp_text, phrase)
        if wer == wer:
            wer_list.append(wer)
        print(f"[HYP TEXT] {hyp_text}")
        print(f"[TRIAL WER] {wer:.4f}" if wer == wer else "[TRIAL WER] nan")
    else:
        print("[HYP TEXT] [skip: phrase label indisponible]")
        print("[TRIAL WER] [skip: phrase label indisponible]")

    # Enregistre un dump detaille pour analyse offline.
    trial_record = {
        "global_id": item.get("global_id"),
        "file": os.path.basename(item["file_path"]),
        "file_path": item["file_path"],
        "trial_key": item.get("trial_key"),
        "session": session,
        "block_num": block_num,
        "trial_num": trial_num,
        "phrase_label": phrase,
        "targets": targets,
        "ensemble_phonemes": ensemble_phonemes,
        "ensemble_words": ensemble_words,
        "per": per,
        "wer": wer if wer == wer else None,
        "models": [
            {
                "name": model_names[i],
                "pred_ids": pred_sequences[i],
                "word_list": model_word_lists[i],
            }
            for i in range(len(model_names))
        ],
    }
    results_dump.append(trial_record)

# Resume global des metriques.
if per_list:
    mean_per = float(np.nanmean(per_list))
    print(f"[ENSEMBLE] Trials evalues: {len(per_list)}")
    print(f"[ENSEMBLE] PER moyen = {mean_per:.4f}")
else:
    print("[ENSEMBLE] Aucun PER calcule.")

if wer_list:
    mean_wer = float(np.nanmean(wer_list))
    print(f"[ENSEMBLE] WER moyen = {mean_wer:.4f} (calcule sur {len(wer_list)} phrases)")
else:
    print("[ENSEMBLE] Aucun WER calcule.")

# Sauvegarde les resultats au format jsonl.
if results_dump:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_path = os.path.join(str(RUNOFF_DIR), f"ensemble_results_{ts}.jsonl")
    with open(out_path, "w", encoding="utf-8") as f_out:
        for rec in results_dump:
            f_out.write(json.dumps(rec, ensure_ascii=True))
            f_out.write("")
    print(f"[SAVE] Resultats ecrits dans {out_path}")
else:
    print("[SAVE] Aucun resultat a ecrire.")


[EVAL] data_val.hdf5 -> 35 trials
[EVAL] data_val.hdf5 -> 49 trials
[EVAL] data_val.hdf5 -> 48 trials
[EVAL] data_val.hdf5 -> 25 trials
[EVAL] data_val.hdf5 -> 25 trials
[EVAL] data_val.hdf5 -> 49 trials
[EVAL] data_val.hdf5 -> 34 trials
[EVAL] Total trials : 265
[ENSEMBLE MODE] wordwise contextuel (vote par mot avec lexique + contexte)
TRIAL global_id=0 | file=data_val.hdf5 | key=trial_0000 | session=t15.2023.08.13 | block_num=8 | trial_num=0
PHRASE LABEL: You can see the code at this point as well.
Targets (len=36): [37, 34, 40, 20, 2, 23, 40, 29, 18, 40, 10, 3, 40, 20, 25, 9, 40, 2, 31, 40, 10, 17, 29, 40, 27, 26, 23, 31, 40, 2, 38, 40, 36, 11, 21, 40]
[MODEL pretrained1] len=38 -> preds=[37, 34, 40, 20, 2, 23, 40, 29, 18, 40, 10, 3, 40, 20, 25, 21, 9, 40, 2, 31, 40, 10, 17, 29, 40, 27, 26, 23, 31, 40, 2, 29, 20, 40, 36, 11, 21, 40]
[MODEL pretrained2] len=37 -> preds=[37, 34, 40, 20, 2, 23, 40, 29, 18, 40, 10, 3, 40, 20, 25, 21, 9, 40, 2, 31, 40, 10, 17, 29, 40, 27, 26, 23, 31, 40,